In [1]:
import os
import pandas as pd
import requests

In [2]:
#FIREBASE
from firebase_admin import credentials,firestore,initialize_app
from google.cloud.firestore_v1.base_query import FieldFilter
from collections import defaultdict

In [3]:
filename=os.environ['FIREBASE_FILENAME']
cred = credentials.Certificate(filename)
firebase = initialize_app(cred)

db = firestore.client()

In [4]:
OANACCOUNT = "5Tv2u4n8BReebmKUNIuN"

In [5]:
donors = db.collection('donors'
                       ).where(filter=FieldFilter("context.account", "==", OANACCOUNT)
                               ).stream()                 

In [6]:
l_donors = []
for donorF in donors:
    #print(payout)
    l_donors.append(donorF.to_dict())


In [7]:
donations = db.collection('donations'
                       ).where(filter=FieldFilter("context.account", "==", OANACCOUNT)
                               ).stream()        

In [8]:
l_donations = []
for donorF in donations:
    #print(payout)
    l_donations.append(donorF.to_dict())

In [9]:
donations_by_donor = defaultdict(list)
for donation in l_donations:
    # Filter donations for the year 2024
    if donation['executedAt'].startswith("2024"):
        donations_by_donor[donation['donor']].append(donation)

In [10]:
#Create a list of donors who donated in 2024
donors_with_donations_2024 = []
for donor_id, donations in donations_by_donor.items():
    donor = next((d for d in l_donors if d['context']['id'] == donor_id and d['nature'] == "naturalPerson"), None)
    #donor = next((d for d in l_donors if d['context']['id'] == donor_id), None)
    if donor:
        total_donated = sum(d['amount'] for d in donations)
        if total_donated >7:
            donors_with_donations_2024.append({
                'donor': donor,
                'total_donated': total_donated
            })

In [11]:
# Sort donors by total donation amount in descending order
donors_with_donations_2024.sort(key=lambda x: x['total_donated'], reverse=True)

In [ ]:
len(donors_with_donations_2024)

In [13]:
# API URL
api_url = os.environ['CERTIFICATEAPI_URL']
l_responses = []

In [ ]:
# Process the list of donors with donations in 2024
for entry in donors_with_donations_2024:
    donor = entry['donor']
    total_donated = entry['total_donated']
    
    # Determine document type based on nature
    document_type = "CIF" if donor['nature'] == "legalPerson" else "DNI"

    # Prepare data for API
    payload = {
        "apiKey": os.environ['CERTIFICATEAPI_APIKEY'],
        "nombre": donor['name'],
        "apellido": donor['lastName'],
        "documento": document_type,
        "n_documento": donor.get('dni', ""),
        "recurrente": donor.get('isRecurrent', ""),
        "cantidad": total_donated,
        "ano": 2024,
        "provincia": donor.get('province', ""),
        "fecha_doc": "17 de enero de 2025",
        "email": donor['email'],
        "enviarCorreo":True
    }

    # Send API request
    response = requests.post(api_url, json=payload)
    j_response = response.json()
    fileUrl = j_response.get('fileUrl','')
    if response.status_code == 200:
        print(f"{donor['name']} {donor['lastName']}: {total_donated} - {fileUrl if fileUrl!='' else 'sin archivo'}.")
    d_response = {
        'donorId':donor['context']['id'],
        'fileUrl': fileUrl
    }
    l_responses.append(d_response)

In [141]:
pd.DataFrame(l_responses).to_csv('certificados2024.csv')